# Health & Personal-Care Hybrid Recommendation System

This notebook contains the same **multi-channel retrieval** implementation as the current retrieval-fix project, reorganized into small, explainable Colab cells. Each major stage has a short description and its own code cell so you can explain it during an interview.

**Pipeline:** load/clean data → per-user temporal split → BPR matrix factorization → semantic product embeddings → two-tower training → FAISS indexes → multi-channel candidate retrieval → learning-to-rank → cross-encoder reranking → safety filtering + MMR → offline evaluation.

**Before you start:** Put `Health_and_Personal_Care.jsonl` and `meta_Health_and_Personal_Care.jsonl` in `MyDrive/HealthRecommender`. Run cells from top to bottom. A full run can take a while because it trains models and reranks candidates for 2,540 validation users and 2,540 test users.

**Important interpretation:** this is an offline portfolio prototype. The safety filter is a basic keyword guardrail, not medical validation, and no live A/B experiment was run. Use validation results when changing settings; the test set is for final evaluation.

## 1. Install dependencies

The notebook uses PyTorch for collaborative models, Sentence Transformers for product text embeddings and cross-encoder reranking, FAISS for vector retrieval, and scikit-learn for the learning-to-rank classifier.

In [ ]:
%pip -q install -U "sentence-transformers>=3.0" "faiss-cpu>=1.8" "scikit-learn>=1.5" "joblib>=1.4" "tqdm>=4.66"

## 2. Mount Google Drive and copy the datasets

Drive is persistent storage. Colab’s `/content` folder is temporary, so copy the files into it each time the runtime resets. This cell checks that both expected files exist before continuing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

DRIVE_FOLDER = Path('/content/drive/MyDrive/HealthRecommender')
REVIEWS_PATH = Path('/content/Health_and_Personal_Care.jsonl')
METADATA_PATH = Path('/content/meta_Health_and_Personal_Care.jsonl')
OUTPUT_DIR = Path('/content/artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, destination in [
    ('Health_and_Personal_Care.jsonl', REVIEWS_PATH),
    ('meta_Health_and_Personal_Care.jsonl', METADATA_PATH),
]:
    source = DRIVE_FOLDER / filename
    if not source.exists():
        raise FileNotFoundError(
            f'File not found: {source}. Upload {filename} to MyDrive/HealthRecommender.'
        )
    shutil.copy2(source, destination)
    print(f'{filename}: {destination.stat().st_size / (1024**2):.1f} MB copied')

print('Output directory:', OUTPUT_DIR)

## 3. Imports, reproducibility, and configuration

A fixed random seed makes stochastic parts of the pipeline more repeatable. `Config` centralizes thresholds, embedding sizes, training epochs, retrieval/ranking cutoffs, score weights, and safety keywords. The current retrieval-fix defaults use up to 200 candidates, then 20 ranked candidates, then 10 final recommendations.

In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
class Config:
    min_user_interactions: int = 3
    min_item_interactions: int = 2
    embedding_dim: int = 64
    mf_epochs: int = 8
    two_tower_epochs: int = 8
    batch_size: int = 512
    negatives_per_positive: int = 3
    learning_rate: float = 2e-3
    weight_decay: float = 1e-5
    retrieval_k: int = 200
    ranking_k: int = 20
    final_k: int = 10
    # Preserve a popularity-based candidate path while retaining personalized retrieval.
    popularity_candidates: int = 30
    ranker_score_weight: float = 0.45
    popularity_score_weight: float = 0.35
    semantic_score_weight: float = 0.15
    cf_score_weight: float = 0.05
    cross_encoder_weight: float = 0.25
    mmr_diversity: float = 0.05
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    safety_block_terms: tuple = (
        "suicide", "self harm", "self-harm", "kill yourself",
        "illegal drug", "methamphetamine", "fentanyl", "weapon",
    )

FEATURE_COLS = [
    "cf_score", "bpr_score", "semantic_score", "pop_score", "log_item_count",
    "item_rating", "verified_rate", "helpful_mean", "user_count",
    "semantic_novelty", "safe",
]

In [ ]:
cfg = Config(device=DEVICE)
print("Configuration:")
print(json.dumps(asdict(cfg), indent=2))

## 4. Data preprocessing

The review file gives user–product interactions and fields such as rating, timestamp, verified purchase, and helpful votes. The metadata file supplies product titles, features, descriptions, and store labels. Preprocessing removes rows missing essential identifiers/timestamps, applies minimum interaction counts, keeps the latest record for each user–product pair, and joins records to products with metadata.

In [ ]:
def read_jsonl(path: str) -> Iterable[dict]:
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_no} of {path}: {exc}") from exc

def load_data(review_path: str, meta_path: str, cfg: Config) -> Tuple[pd.DataFrame, dict]:
    print("Reading review records...")
    rows = []
    for r in tqdm(read_jsonl(review_path), desc="Reviews"):
        user = r.get("user_id")
        item = r.get("parent_asin")
        ts = r.get("timestamp")
        if user is None or item is None or ts is None:
            continue
        try:
            rating = float(r.get("rating", 0.0) or 0.0)
            timestamp = int(ts)
            helpful = float(r.get("helpful_vote", 0) or 0)
        except (ValueError, TypeError):
            continue
        rows.append({
            "user": str(user), "item": str(item), "rating": rating,
            "timestamp": timestamp,
            "verified": int(bool(r.get("verified_purchase", False))),
            "helpful": helpful,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No usable review records were found. Check the review file and column names.")

    print(f"Raw usable review rows: {len(df):,}")
    user_counts = df["user"].value_counts()
    item_counts = df["item"].value_counts()
    df = df[
        df["user"].map(user_counts).ge(cfg.min_user_interactions)
        & df["item"].map(item_counts).ge(cfg.min_item_interactions)
    ].copy()

    # Keep the latest review for each user-product pair. This prevents a repeated
    # review from being treated as multiple distinct products in that user's history.
    df = (df.sort_values(["user", "item", "timestamp"])
            .drop_duplicates(["user", "item"], keep="last")
            .sort_values("timestamp")
            .reset_index(drop=True))

    print("Reading product metadata...")
    metadata = {}
    for r in tqdm(read_jsonl(meta_path), desc="Metadata"):
        raw_item = r.get("parent_asin")
        if raw_item is None:
            continue
        item = str(raw_item)
        title = str(r.get("title") or "")
        features = r.get("features") or []
        description = r.get("description") or []

        def list_to_text(value):
            if isinstance(value, list):
                return " ".join(str(v) for v in value)
            return str(value)

        text = " ".join([
            title,
            list_to_text(features),
            list_to_text(description),
            str(r.get("store") or ""),
        ])
        metadata[item] = {
            "title": title or item,
            "text": re.sub(r"\s+", " ", text).strip(),
            "store": str(r.get("store") or ""),
        }

    before = len(df)
    df = df[df["item"].isin(metadata)].copy()
    if df.empty:
        raise ValueError("No overlap was found between review product IDs and metadata product IDs.")
    print(f"Rows without matching metadata removed: {before - len(df):,}")
    return df.reset_index(drop=True), metadata

In [ ]:
df, metadata = load_data(str(REVIEWS_PATH), str(METADATA_PATH), cfg)
print(f"Cleaned interactions: {len(df):,}")
print(f"Unique users: {df['user'].nunique():,}")
print(f"Unique products: {df['item'].nunique():,}")
display(df.head())

## 5. Train / validation / test split

We use a **per-user temporal split**: for each user, all but their last two distinct interactions go to training, the second-last goes to validation, and the last goes to testing. This simulates predicting a later interaction from that user’s earlier history. We then assemble one global training set, one validation set, and one test set.

In [ ]:
def time_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Per user: all but last two interactions train; second-last validation; last test."""
    train_parts, val_rows, test_rows = [], [], []
    for user, group in df.groupby("user", sort=False):
        group = group.sort_values("timestamp")
        if len(group) < 3:
            continue
        train_parts.append(group.iloc[:-2])
        val_rows.append(group.iloc[-2])
        test_rows.append(group.iloc[-1])
    if not train_parts or not val_rows or not test_rows:
        raise ValueError("Temporal split created an empty set. Try reviewing interaction thresholds.")
    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.DataFrame(val_rows).reset_index(drop=True)
    test_df = pd.DataFrame(test_rows).reset_index(drop=True)
    return train_df, val_df, test_df

class IDMaps:
    def __init__(self, train_df: pd.DataFrame, catalog_items: Sequence[str] | None = None):
        self.id2user = sorted(train_df["user"].unique().tolist())
        # Collaborative item IDs cover training items only; the semantic catalogue
        # also includes items seen only in validation/test, enabling content cold start.
        self.id2item = sorted(train_df["item"].unique().tolist())
        self.user2id = {u: i for i, u in enumerate(self.id2user)}
        self.item2id = {it: i for i, it in enumerate(self.id2item)}
        chosen_catalog = self.id2item if catalog_items is None else catalog_items
        self.id2catalog_item = sorted(set(str(it) for it in chosen_catalog))
        self.catalog2id = {it: i for i, it in enumerate(self.id2catalog_item)}

def popularity_stats(train_df: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame, float, float]:
    stats = train_df.groupby("item").agg(
        count=("rating", "size"),
        rating_sum=("rating", "sum"),
        mean_rating=("rating", "mean"),
        verified_sum=("verified", "sum"),
        helpful_sum=("helpful", "sum"),
    )
    global_mean = float(train_df["rating"].mean())
    m = max(5.0, float(stats["count"].quantile(0.75)))
    stats["score"] = (
        (stats["count"] / (stats["count"] + m)) * stats["mean_rating"]
        + (m / (stats["count"] + m)) * global_mean
    )
    return stats["score"].sort_values(ascending=False), stats, global_mean, m

def build_histories(train_df: pd.DataFrame) -> Dict[str, List[str]]:
    histories = defaultdict(list)
    for row in train_df.sort_values("timestamp").itertuples(index=False):
        histories[row.user].append(row.item)
    return dict(histories)

In [ ]:
train_df, val_df, test_df = time_split(df)
catalog_items = sorted(df["item"].unique().tolist())
ids = IDMaps(train_df, catalog_items=catalog_items)

val_before, test_before = len(val_df), len(test_df)
val_df = val_df[val_df.item.isin(ids.catalog2id)].reset_index(drop=True)
test_df = test_df[test_df.item.isin(ids.catalog2id)].reset_index(drop=True)

print(f"Train interactions: {len(train_df):,}")
print(f"Validation targets retained: {len(val_df):,}/{val_before:,}")
print(f"Test targets retained: {len(test_df):,}/{test_before:,}")
print(f"Training users: {len(ids.id2user):,}")
print(f"Training items: {len(ids.id2item):,}")
print(f"Semantic candidate catalogue: {len(ids.id2catalog_item):,}")

popularity, item_stats, global_mean, smoothing_m = popularity_stats(train_df)
histories = build_histories(train_df)

## 6. BPR matrix factorization (collaborative filtering)

BPR means Bayesian Personalized Ranking. It learns user and item vectors so an observed interaction receives a higher score than an item sampled from the user’s unobserved products. In this implementation, every observed review is treated as positive regardless of its star rating; an unobserved item is only a sampled negative, not a confirmed dislike.

In [ ]:
class BPRMatrixFactorization(nn.Module):
    def __init__(self, n_users: int, n_items: int, dim: int):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)

    def score(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(dim=-1)

def train_bpr(train_df: pd.DataFrame, ids: IDMaps, cfg: Config):
    pairs = [(ids.user2id[r.user], ids.item2id[r.item]) for r in train_df.itertuples(index=False)]
    positives = defaultdict(set)
    for user_id, item_id in pairs:
        positives[user_id].add(item_id)
    model = BPRMatrixFactorization(len(ids.id2user), len(ids.id2item), cfg.embedding_dim).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    all_item_ids = np.arange(len(ids.id2item))
    negative_candidates = {u: np.asarray([i for i in all_item_ids if i not in seen], dtype=np.int64)
                           for u, seen in positives.items()}

    model.train()
    for epoch in range(cfg.mf_epochs):
        random.shuffle(pairs)
        total_loss = 0.0
        used = 0
        for start in range(0, len(pairs), cfg.batch_size):
            batch = pairs[start:start + cfg.batch_size]
            valid = [(u, i) for u, i in batch if len(negative_candidates[u]) > 0]
            if not valid:
                continue
            us = torch.tensor([p[0] for p in valid], dtype=torch.long, device=cfg.device)
            pos = torch.tensor([p[1] for p in valid], dtype=torch.long, device=cfg.device)
            neg_ids = [int(random.choice(negative_candidates[u])) for u, _ in valid]
            neg = torch.tensor(neg_ids, dtype=torch.long, device=cfg.device)
            pos_score, neg_score = model.score(us, pos), model.score(us, neg)
            loss = -F.logsigmoid(pos_score - neg_score).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * len(valid)
            used += len(valid)
        print(f"BPR epoch {epoch + 1}/{cfg.mf_epochs} | loss={total_loss / max(used, 1):.4f}")
    return model

In [ ]:
bpr = train_bpr(train_df, ids, cfg)
torch.save(bpr.state_dict(), OUTPUT_DIR / "bpr_mf.pt")
with torch.no_grad():
    bpr_user_cf = F.normalize(bpr.user_emb.weight, dim=-1).cpu().numpy().astype("float32")
    bpr_train_item_cf = F.normalize(bpr.item_emb.weight, dim=-1).cpu().numpy().astype("float32")
print("BPR user vectors:", bpr_user_cf.shape)
print("BPR item vectors:", bpr_train_item_cf.shape)

## 7. Semantic product embeddings

A pretrained Sentence Transformer encodes each product’s title, features, description, and store label into a 384-dimensional vector. Unlike collaborative filtering, these vectors represent product text. The vectors are cached in the artifacts directory and reused when the item ordering matches.

In [ ]:
def build_semantic_embeddings(item_ids: Sequence[str], metadata: dict, model_name: str,
                              output_dir: Path, device: str, force_rebuild: bool = False) -> np.ndarray:
    from sentence_transformers import SentenceTransformer

    cache_path = output_dir / "semantic_embeddings.npy"
    ids_path = output_dir / "semantic_item_ids.json"
    if not force_rebuild and cache_path.exists() and ids_path.exists():
        try:
            cached_ids = json.loads(ids_path.read_text(encoding="utf-8"))
            cached = np.load(cache_path)
            if cached_ids == list(item_ids) and cached.shape[0] == len(item_ids):
                print("Using cached semantic embeddings.")
                return cached.astype("float32")
        except Exception:
            pass

    texts = []
    for item in item_ids:
        text = metadata[item]["text"][:2000]
        texts.append(text if text.strip() else metadata[item]["title"])
    encoder = SentenceTransformer(model_name, device=device)
    emb = encoder.encode(
        texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    np.save(cache_path, emb)
    ids_path.write_text(json.dumps(list(item_ids)), encoding="utf-8")
    return emb

In [ ]:
semantic = build_semantic_embeddings(
    ids.id2catalog_item,
    metadata,
    cfg.embedding_model,
    OUTPUT_DIR,
    cfg.device,
)
print("Semantic embedding matrix:", semantic.shape)

## 8. Two-tower retrieval model

The two-tower model has a user embedding table and an item embedding table. Each maps an ID to a learned 64-dimensional vector; normalized dot products score user–item pairs. It is trained with observed interactions and sampled unobserved items. In this implementation these towers are ID lookups, not deep text encoders.

In [ ]:
class TwoTower(nn.Module):
    """ID-embedding user tower and ID-embedding item tower."""
    def __init__(self, n_users: int, n_items: int, dim: int):
        super().__init__()
        self.user = nn.Embedding(n_users, dim)
        self.item = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user.weight, std=0.05)
        nn.init.normal_(self.item.weight, std=0.05)

    def forward(self, users, items):
        user_vec = F.normalize(self.user(users), dim=-1)
        item_vec = F.normalize(self.item(items), dim=-1)
        return (user_vec * item_vec).sum(dim=-1)

def train_two_tower(train_df: pd.DataFrame, ids: IDMaps, cfg: Config):
    pairs = [(ids.user2id[r.user], ids.item2id[r.item]) for r in train_df.itertuples(index=False)]
    positives = defaultdict(set)
    for user_id, item_id in pairs:
        positives[user_id].add(item_id)
    all_item_ids = np.arange(len(ids.id2item))
    negative_candidates = {u: np.asarray([i for i in all_item_ids if i not in seen], dtype=np.int64)
                           for u, seen in positives.items()}

    model = TwoTower(len(ids.id2user), len(ids.id2item), cfg.embedding_dim).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    model.train()
    for epoch in range(cfg.two_tower_epochs):
        random.shuffle(pairs)
        total_loss, used = 0.0, 0
        for start in range(0, len(pairs), cfg.batch_size):
            batch = [(u, i) for u, i in pairs[start:start + cfg.batch_size]
                     if len(negative_candidates[u]) > 0]
            if not batch:
                continue
            us = torch.tensor([x[0] for x in batch], dtype=torch.long, device=cfg.device)
            pos = torch.tensor([x[1] for x in batch], dtype=torch.long, device=cfg.device)
            neg_rows = []
            for u, _ in batch:
                candidates = negative_candidates[u]
                replace = len(candidates) < cfg.negatives_per_positive
                neg_rows.append(np.random.choice(candidates, size=cfg.negatives_per_positive, replace=replace))
            neg = torch.tensor(np.asarray(neg_rows), dtype=torch.long, device=cfg.device)
            uvec = F.normalize(model.user(us), dim=-1)
            pvec = F.normalize(model.item(pos), dim=-1)
            nvec = F.normalize(model.item(neg), dim=-1)
            pos_logits = (uvec * pvec).sum(-1, keepdim=True)
            neg_logits = torch.einsum("bd,bkd->bk", uvec, nvec)
            logits = torch.cat([pos_logits, neg_logits], dim=1) / 0.07
            labels = torch.zeros(len(batch), dtype=torch.long, device=cfg.device)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += float(loss.detach()) * len(batch)
            used += len(batch)
        print(f"Two-tower epoch {epoch + 1}/{cfg.two_tower_epochs} | loss={total_loss / max(used, 1):.4f}")
    return model

In [ ]:
two_tower = train_two_tower(train_df, ids, cfg)
torch.save(two_tower.state_dict(), OUTPUT_DIR / "two_tower.pt")
with torch.no_grad():
    user_cf = F.normalize(two_tower.user.weight, dim=-1).cpu().numpy().astype("float32")
    trained_item_cf = F.normalize(two_tower.item.weight, dim=-1).cpu().numpy().astype("float32")

# Products in the full semantic catalogue but absent from training get zero collaborative vectors.
item_cf = np.zeros((len(ids.id2catalog_item), cfg.embedding_dim), dtype=np.float32)
bpr_item_cf = np.zeros((len(ids.id2catalog_item), cfg.embedding_dim), dtype=np.float32)
for train_item, train_idx in ids.item2id.items():
    catalog_idx = ids.catalog2id[train_item]
    item_cf[catalog_idx] = trained_item_cf[train_idx]
    bpr_item_cf[catalog_idx] = bpr_train_item_cf[train_idx]

print("Two-tower user vectors:", user_cf.shape)
print("Two-tower catalogue item vectors:", item_cf.shape)

## 9. FAISS indexes and multi-channel retrieval

FAISS indexes vectors for fast similarity search. This notebook builds (1) a hybrid HNSW approximate-nearest-neighbor index combining collaborative and semantic item vectors, and (2) an exact semantic index. Candidate generation later also includes a popularity channel; the three lists are fused using reciprocal rank fusion (RRF).

In [ ]:
def normalize_rows(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)

def build_faiss_index(item_cf: np.ndarray, item_semantic: np.ndarray, output_path: Path):
    import faiss
    cf = normalize_rows(item_cf)
    sem = normalize_rows(item_semantic)
    hybrid = np.concatenate([0.6 * cf, 0.4 * sem], axis=1).astype("float32")
    hybrid = normalize_rows(hybrid)
    index = faiss.IndexHNSWFlat(hybrid.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 80
    index.hnsw.efSearch = 64
    index.add(hybrid)
    faiss.write_index(index, str(output_path))
    return index, hybrid

def build_semantic_faiss_index(item_semantic: np.ndarray, output_path: Path):
    """Build an exact cosine-similarity index for content-only retrieval."""
    import faiss
    sem = normalize_rows(item_semantic).astype("float32")
    index = faiss.IndexFlatIP(sem.shape[1])
    index.add(sem)
    faiss.write_index(index, str(output_path))
    return index

def normalize_vector(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)
    return v / max(float(np.linalg.norm(v)), 1e-8)

In [ ]:
index, hybrid_item_vectors = build_faiss_index(
    item_cf, semantic, OUTPUT_DIR / "hybrid_hnsw.faiss"
)
semantic_index = build_semantic_faiss_index(
    semantic, OUTPUT_DIR / "semantic_flat.faiss"
)
print("Hybrid index items:", index.ntotal)
print("Semantic index items:", semantic_index.ntotal)

## 10. Learning-to-rank

The ranker uses features such as two-tower similarity, BPR score, semantic match, popularity, rating statistics, verified-purchase rate, helpful votes, user-history length, novelty, and safety. A pointwise `HistGradientBoostingClassifier` learns to separate observed training items from sampled unobserved candidates. This is a pointwise ranker, not LambdaMART/listwise ranking.

In [ ]:
class FeatureBuilder:
    """Vectorized pointwise ranking features; dataset-wide stats are cached once."""
    def __init__(self, train_df, ids, metadata, semantic, user_cf, item_cf,
                 bpr_user_cf, bpr_item_cf, pop_scores,
                 item_stats, global_mean, smoothing_m, cfg):
        self.train_df = train_df
        self.ids = ids
        self.metadata = metadata
        self.semantic = semantic
        self.user_cf = user_cf
        self.item_cf = item_cf
        self.bpr_user_cf = bpr_user_cf
        self.bpr_item_cf = bpr_item_cf
        self.pop_scores = pop_scores
        self.item_stats = item_stats
        self.global_mean = global_mean
        self.smoothing_m = smoothing_m
        self.cfg = cfg
        self.user_count = train_df["user"].value_counts().to_dict()
        self.verified_total = float(train_df["verified"].sum())
        self.helpful_total = float(train_df["helpful"].sum())
        self.rating_total = float(train_df["rating"].sum())
        self.total_rows = max(len(train_df), 1)
        self.item_count = item_stats["count"].to_dict()
        self.item_rating_sum = item_stats["rating_sum"].to_dict()
        self.item_verified_sum = item_stats["verified_sum"].to_dict()
        self.item_helpful_sum = item_stats["helpful_sum"].to_dict()
        self.safe_map = {
            item: not any(term in metadata[item]["text"].lower() for term in cfg.safety_block_terms)
            for item in ids.id2catalog_item
        }

    def user_profile(self, history_items: Sequence[str]) -> np.ndarray:
        idx = [self.ids.catalog2id[x] for x in history_items if x in self.ids.catalog2id]
        if not idx:
            return np.zeros(self.semantic.shape[1], dtype=np.float32)
        return normalize_vector(self.semantic[idx].mean(axis=0))

    def batch(self, users: Sequence[str], items: Sequence[str], profiles: np.ndarray,
              exclude_observation: np.ndarray | None = None,
              target_ratings: np.ndarray | None = None,
              target_verified: np.ndarray | None = None,
              target_helpful: np.ndarray | None = None) -> pd.DataFrame:
        n = len(items)
        if n == 0:
            return pd.DataFrame(columns=FEATURE_COLS + ["item"])
        exclude = np.zeros(n, dtype=bool) if exclude_observation is None else np.asarray(exclude_observation, dtype=bool)
        target_ratings = np.zeros(n, dtype=np.float32) if target_ratings is None else np.asarray(target_ratings, dtype=np.float32)
        target_verified = np.zeros(n, dtype=np.float32) if target_verified is None else np.asarray(target_verified, dtype=np.float32)
        target_helpful = np.zeros(n, dtype=np.float32) if target_helpful is None else np.asarray(target_helpful, dtype=np.float32)
        user_indices = np.asarray([self.ids.user2id.get(u, -1) for u in users], dtype=np.int64)
        item_indices = np.asarray([self.ids.catalog2id.get(it, -1) for it in items], dtype=np.int64)
        if np.any(item_indices < 0):
            raise ValueError("Feature generation received an item outside the training catalogue.")

        user_vecs = np.zeros((n, self.user_cf.shape[1]), dtype=np.float32)
        bpr_user_vecs = np.zeros((n, self.bpr_user_cf.shape[1]), dtype=np.float32)
        known_users = user_indices >= 0
        user_vecs[known_users] = self.user_cf[user_indices[known_users]]
        bpr_user_vecs[known_users] = self.bpr_user_cf[user_indices[known_users]]
        candidate_cf = self.item_cf[item_indices]
        candidate_bpr = self.bpr_item_cf[item_indices]
        candidate_sem = self.semantic[item_indices]
        cf_score = np.sum(user_vecs * candidate_cf, axis=1)
        bpr_score = np.sum(bpr_user_vecs * candidate_bpr, axis=1)
        sem_score = np.sum(np.asarray(profiles, dtype=np.float32) * candidate_sem, axis=1)

        raw_counts = np.asarray([self.item_count.get(it, 0) for it in items], dtype=np.float32)
        rating_sums = np.asarray([self.item_rating_sum.get(it, 0.0) for it in items], dtype=np.float32)
        verified_sums = np.asarray([self.item_verified_sum.get(it, 0.0) for it in items], dtype=np.float32)
        helpful_sums = np.asarray([self.item_helpful_sum.get(it, 0.0) for it in items], dtype=np.float32)
        excl = exclude.astype(np.float32)
        adjusted_counts = np.maximum(raw_counts - excl, 0.0)
        adjusted_rating_sums = rating_sums - target_ratings * excl
        adjusted_verified_sums = verified_sums - target_verified * excl
        adjusted_helpful_sums = helpful_sums - target_helpful * excl

        adjusted_global_mean = (self.rating_total - target_ratings * excl) / np.maximum(self.total_rows - excl, 1.0)
        default_rating = adjusted_global_mean
        item_rating = np.where(adjusted_counts > 0, adjusted_rating_sums / np.maximum(adjusted_counts, 1), default_rating)
        verified_rate = np.where(adjusted_counts > 0, adjusted_verified_sums / np.maximum(adjusted_counts, 1), 0.0)
        helpful_mean = np.where(adjusted_counts > 0, adjusted_helpful_sums / np.maximum(adjusted_counts, 1), 0.0)
        pop_score = (
            (adjusted_counts / np.maximum(adjusted_counts + self.smoothing_m, 1e-8)) * item_rating
            + (self.smoothing_m / np.maximum(adjusted_counts + self.smoothing_m, 1e-8)) * adjusted_global_mean
        )
        user_counts = np.asarray([self.user_count.get(u, 0) for u in users], dtype=np.float32) - excl
        safe = np.asarray([float(self.safe_map.get(it, False)) for it in items], dtype=np.float32)
        frame = pd.DataFrame({
            "item": list(items),
            "cf_score": cf_score,
            "bpr_score": bpr_score,
            "semantic_score": sem_score,
            "pop_score": pop_score,
            "log_item_count": np.log1p(adjusted_counts),
            "item_rating": item_rating,
            "verified_rate": verified_rate,
            "helpful_mean": helpful_mean,
            "user_count": np.maximum(user_counts, 0.0),
            "semantic_novelty": 1.0 - sem_score,
            "safe": safe,
        })
        return frame.replace([np.inf, -np.inf], 0.0).fillna(0.0)

def train_ranker(train_df, ids, feature_builder, semantic, cfg):
    """Train a pointwise ranker using observed items and sampled unobserved items."""
    rng = np.random.default_rng(SEED)
    positive_by_user = defaultdict(set)
    for row in train_df.itertuples(index=False):
        positive_by_user[row.user].add(row.item)
    histories = build_histories(train_df)
    profile_sums = {}
    profile_counts = {}
    for user, items in histories.items():
        idx = [ids.catalog2id[x] for x in items if x in ids.catalog2id]
        profile_sums[user] = semantic[idx].sum(axis=0) if idx else np.zeros(semantic.shape[1], dtype=np.float32)
        profile_counts[user] = len(idx)

    sample_users, sample_items, sample_profiles = [], [], []
    sample_exclude, sample_ratings, sample_verified, sample_helpful, labels = [], [], [], [], []
    all_items = np.asarray(ids.id2item, dtype=object)
    negative_pool_by_user = {
        user: np.asarray([it for it in all_items if it not in positive_items], dtype=object)
        for user, positive_items in positive_by_user.items()
    }
    # Sample some difficult negatives from popular items, not only easy random items.
    # This better matches the candidate distribution encountered at inference.
    popular_pool = np.asarray(
        [str(it) for it in feature_builder.pop_scores.index[:500] if str(it) in ids.item2id],
        dtype=object,
    )

    for row in train_df.itertuples(index=False):
        user, item = row.user, row.item
        item_idx = ids.catalog2id[item]
        leave_one_out = profile_sums[user] - semantic[item_idx]
        if profile_counts[user] > 1:
            leave_one_out = normalize_vector(leave_one_out)
        else:
            leave_one_out = np.zeros(semantic.shape[1], dtype=np.float32)
        sample_users.append(user); sample_items.append(item); sample_profiles.append(leave_one_out)
        sample_exclude.append(True); sample_ratings.append(float(row.rating))
        sample_verified.append(float(row.verified)); sample_helpful.append(float(row.helpful)); labels.append(1)

        candidates = negative_pool_by_user.get(user, np.asarray([], dtype=object))
        if len(candidates) == 0:
            continue
        negative_count = min(cfg.negatives_per_positive, len(candidates))
        candidate_set = set(candidates.tolist())
        hard_pool = np.asarray([it for it in popular_pool if it in candidate_set], dtype=object)
        hard_count = min(len(hard_pool), max(1, negative_count // 2)) if negative_count else 0
        hard_negs = rng.choice(hard_pool, size=hard_count, replace=False).tolist() if hard_count else []
        remaining_pool = np.asarray([it for it in candidates if it not in set(hard_negs)], dtype=object)
        random_count = min(negative_count - len(hard_negs), len(remaining_pool))
        random_negs = rng.choice(remaining_pool, size=random_count, replace=False).tolist() if random_count else []
        negs = hard_negs + random_negs
        # Use the same leave-one-out user context for the positive and its negatives.
        for neg_item in negs:
            sample_users.append(user); sample_items.append(str(neg_item)); sample_profiles.append(leave_one_out)
            sample_exclude.append(False); sample_ratings.append(0.0)
            sample_verified.append(0.0); sample_helpful.append(0.0); labels.append(0)

    profiles = np.asarray(sample_profiles, dtype=np.float32)
    # Leave-one-out aggregate features prevent a positive training example from
    # contributing its own rating/count to item-level features.
    X = feature_builder.batch(
        sample_users, sample_items, profiles,
        np.asarray(sample_exclude), np.asarray(sample_ratings),
        np.asarray(sample_verified), np.asarray(sample_helpful),
    )[FEATURE_COLS]
    y = np.asarray(labels, dtype=np.int64)
    if len(np.unique(y)) < 2:
        raise ValueError(
            "The ranker training set contains only one label class. This usually means the "
            "filtered catalogue is too small to sample unobserved items; reduce filtering thresholds "
            "or check the dataset."
        )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.to_numpy(dtype=np.float32))
    model = HistGradientBoostingClassifier(
        max_iter=120, learning_rate=0.08, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=SEED,
    )
    model.fit(X_scaled, y)
    print(f"Learning-to-rank examples: {len(y):,} ({int(y.sum()):,} positive, {int((y == 0).sum()):,} sampled negative)")
    return model, scaler

In [ ]:
feature_builder = FeatureBuilder(
    train_df, ids, metadata, semantic, user_cf, item_cf,
    bpr_user_cf, bpr_item_cf, popularity,
    item_stats, global_mean, smoothing_m, cfg,
)
ranker, scaler = train_ranker(train_df, ids, feature_builder, semantic, cfg)
joblib.dump(
    {"model": ranker, "scaler": scaler, "feature_cols": FEATURE_COLS},
    OUTPUT_DIR / "ranker.joblib"
)
print("Ranker trained and saved.")

## 11. Candidate retrieval and first-stage ranking

For each evaluation user, retrieve candidates separately from hybrid FAISS, semantic-only FAISS, and popularity, then combine them with reciprocal rank fusion. The code measures candidate recall before ranking. It then creates ranking features, scores the candidates, and keeps the top 20 for reranking. A target missing from these initial candidates cannot be recovered by later stages.

In [ ]:
def is_safe(item: str, metadata: dict, cfg: Config) -> bool:
    txt = metadata.get(item, {}).get("text", "").lower()
    return not any(term in txt for term in cfg.safety_block_terms)

def retrieve_candidates(user: str, history: Sequence[str], ids, tt_model, semantic, index,
                        semantic_index, cfg: Config, popularity, metadata):
    """Fuse hybrid ANN, exact semantic, and popularity candidate channels.

    The earlier single hybrid index mixed collaborative and semantic vectors in
    one similarity score. That can suppress content-relevant candidates,
    especially items with no training interactions. This version searches each
    channel separately, reserves a small popularity pool, and uses reciprocal
    rank fusion to fill the remaining candidate budget.
    """
    seen = set(history)
    retrieval_k = max(1, int(cfg.retrieval_k))
    popular_keep = min(max(10, cfg.popularity_candidates), max(retrieval_k // 3, 1))
    search_k = min(len(ids.id2catalog_item), retrieval_k + len(seen) + 50)

    if user in ids.user2id:
        with torch.no_grad():
            uid = torch.tensor([ids.user2id[user]], dtype=torch.long, device=cfg.device)
            user_cf = F.normalize(tt_model.user(uid), dim=-1).cpu().numpy()[0].astype(np.float32)
    else:
        user_cf = np.zeros(cfg.embedding_dim, dtype=np.float32)

    hist_indices = [ids.catalog2id[x] for x in history if x in ids.catalog2id]
    if hist_indices:
        user_sem = normalize_vector(semantic[hist_indices].mean(axis=0))
    else:
        user_sem = np.zeros(semantic.shape[1], dtype=np.float32)

    # Channel 1: hybrid collaborative + semantic nearest neighbours.
    hybrid_candidates = []
    if user in ids.user2id or np.linalg.norm(user_sem) > 1e-8:
        query = normalize_vector(np.concatenate([0.6 * user_cf, 0.4 * user_sem]).astype(np.float32))
        _, ann_indices = index.search(query.reshape(1, -1), search_k)
        hybrid_candidates = [ids.id2catalog_item[int(i)] for i in ann_indices[0] if i >= 0]
        hybrid_candidates = [it for it in hybrid_candidates
                             if it not in seen and is_safe(it, metadata, cfg)]

    # Channel 2: semantic-only retrieval can find content-similar items even
    # when their collaborative vector is missing or weak.
    semantic_candidates = []
    if np.linalg.norm(user_sem) > 1e-8:
        _, sem_indices = semantic_index.search(user_sem.reshape(1, -1).astype("float32"), search_k)
        semantic_candidates = [ids.id2catalog_item[int(i)] for i in sem_indices[0] if i >= 0]
        semantic_candidates = [it for it in semantic_candidates
                               if it not in seen and is_safe(it, metadata, cfg)]

    # Channel 3: popularity candidates guarantee that the baseline's strongest
    # items can compete after ranking rather than being lost at retrieval.
    popular_candidates = [str(it) for it in popularity.index
                          if str(it) not in seen and is_safe(str(it), metadata, cfg)]

    # Reciprocal rank fusion combines ranks from differently-scaled channels.
    # Using ranks avoids directly comparing raw cosine similarities and ratings.
    rrf_scores = defaultdict(float)
    rrf_constant = 60.0
    for weight, channel in ((1.0, hybrid_candidates), (1.2, semantic_candidates), (0.8, popular_candidates[:search_k])):
        for rank, item in enumerate(channel, start=1):
            rrf_scores[item] += weight / (rrf_constant + rank)

    # Reserve high-popularity candidates explicitly, then fill the rest by RRF.
    reserved_popular = popular_candidates[:popular_keep]
    candidates = list(reserved_popular)
    candidate_set = set(candidates)
    for item, _ in sorted(rrf_scores.items(), key=lambda pair: pair[1], reverse=True):
        if item not in candidate_set:
            candidates.append(item)
            candidate_set.add(item)
        if len(candidates) >= retrieval_k:
            break

    # If safety and seen-item filtering leave too few candidates, backfill from
    # remaining safe catalogue items ordered by popularity then catalogue order.
    if len(candidates) < retrieval_k:
        for item in popular_candidates:
            if item not in candidate_set:
                candidates.append(item)
                candidate_set.add(item)
            if len(candidates) >= retrieval_k:
                break
    channel_candidates = {
        "hybrid": hybrid_candidates[:retrieval_k],
        "semantic": semantic_candidates[:retrieval_k],
        "popularity": popular_candidates[:retrieval_k],
    }
    return candidates[:retrieval_k], channel_candidates

def normalize_scores(scores: Sequence[float]) -> np.ndarray:
    values = np.asarray(scores, dtype=np.float32)
    if len(values) == 0:
        return values
    lo, hi = float(np.min(values)), float(np.max(values))
    if hi - lo < 1e-8:
        if len(values) == 1:
            return np.ones(1, dtype=np.float32)
        return np.linspace(1.0, 0.0, len(values), dtype=np.float32)
    return (values - lo) / (hi - lo)

def prepare_ranked_candidates(eval_df, split_name, histories, ids, tt_model, semantic, index,
                              semantic_index, cfg, popularity, metadata, feature_builder, ranker, scaler,
                              max_eval_users=None):
    eval_df = eval_df.sort_values("user").reset_index(drop=True)
    if max_eval_users is not None:
        eval_df = eval_df.head(max_eval_users).copy()
    result = {}
    profiles_by_user = {}
    candidates_by_user = {}
    print(f"Generating {split_name} candidate lists for {len(eval_df):,} users...")
    candidate_hits = 0
    channel_hits = {"hybrid": 0, "semantic": 0, "popularity": 0}
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc=f"{split_name} retrieval/ranking"):
        history = histories.get(row.user, [])
        candidates, channel_candidates = retrieve_candidates(
            row.user, history, ids, tt_model, semantic, index, semantic_index,
            cfg, popularity, metadata
        )
        if not candidates:
            candidates = [it for it in popularity.index if it not in set(history) and is_safe(it, metadata, cfg)][:cfg.retrieval_k]
        if row.item in candidates:
            candidate_hits += 1
        for channel_name in channel_hits:
            if row.item in channel_candidates[channel_name]:
                channel_hits[channel_name] += 1
        if not candidates:
            result[row.user] = []
            continue
        profile = feature_builder.user_profile(history)
        feat = feature_builder.batch([row.user] * len(candidates), candidates,
                                     np.repeat(profile.reshape(1, -1), len(candidates), axis=0))
        x = scaler.transform(feat[FEATURE_COLS].to_numpy(dtype=np.float32))
        ranker_scores = ranker.predict_proba(x)[:, 1]
        # A normalized multi-signal score blend: learned ranker + popularity prior
        # + user's semantic profile + collaborative similarity. Weights are tunable
        # on validation data; this is not a guarantee of better test metrics.
        blend_scores = (
            cfg.ranker_score_weight * normalize_scores(ranker_scores)
            + cfg.popularity_score_weight * normalize_scores(feat["pop_score"].to_numpy())
            + cfg.semantic_score_weight * normalize_scores(feat["semantic_score"].to_numpy())
            + cfg.cf_score_weight * normalize_scores(feat["cf_score"].to_numpy())
        )
        order = np.argsort(-blend_scores)[:cfg.ranking_k]
        ranked_items = feat.iloc[order]["item"].tolist()
        ranked_scores = blend_scores[order].astype(float).tolist()
        result[row.user] = ranked_items
        profiles_by_user[row.user] = " User interests: " + " | ".join(
            metadata[x]["title"] for x in history[-10:] if x in metadata
        )
        candidates_by_user[row.user] = ranked_scores
    denominator = max(len(eval_df), 1)
    candidate_recall = candidate_hits / denominator
    print(f"Candidate Recall@{cfg.retrieval_k} after multi-channel fusion: {candidate_recall:.4f} ({candidate_hits}/{len(eval_df)})")
    for channel_name, hits in channel_hits.items():
        print(f"  {channel_name.title()} channel Recall@{cfg.retrieval_k}: {hits / denominator:.4f} ({hits}/{len(eval_df)})")
    return eval_df, result, profiles_by_user, candidates_by_user

## 12. Cross-encoder reranking

The cross-encoder compares the user’s recent product-interest text with each candidate’s product text and predicts a relevance score. It reranks the top 20 candidates rather than the whole catalogue. The code blends cross-encoder scores with earlier ranker/popularity/semantic/collaborative signals so that the pretrained text-relevance model does not completely replace them.

In [ ]:
def cross_encoder_rerank_all(ranked_items_by_user, profiles_by_user, metadata, model, batch_size=64):
    """Score all user-candidate text pairs in batches to avoid thousands of tiny model calls."""
    scores_by_user = {u: [] for u in ranked_items_by_user}
    pending_pairs, pending_keys = [], []

    def flush():
        if not pending_pairs:
            return
        predictions = model.predict(pending_pairs, batch_size=batch_size, show_progress_bar=False)
        predictions = np.asarray(predictions).reshape(-1)
        for (user, item), score in zip(pending_keys, predictions):
            scores_by_user[user].append((item, float(score)))
        pending_pairs.clear()
        pending_keys.clear()

    total = sum(len(items) for items in ranked_items_by_user.values())
    print(f"Cross-encoder reranking {total:,} user-item pairs...")
    for user, items in tqdm(ranked_items_by_user.items(), desc="Preparing reranker pairs"):
        profile = profiles_by_user.get(user, " User interests: ")
        for item in items:
            pending_pairs.append((profile[:700], metadata[item]["text"][:1000]))
            pending_keys.append((user, item))
            if len(pending_pairs) >= 2048:
                flush()
    flush()
    return scores_by_user

In [ ]:
cross_encoder = None
try:
    from sentence_transformers import CrossEncoder
    print(f"Loading cross-encoder: {cfg.cross_encoder_model} on {cfg.device}")
    cross_encoder = CrossEncoder(cfg.cross_encoder_model, device=cfg.device)
    print("Cross-encoder loaded.")
except Exception as exc:
    print("WARNING: Cross-encoder unavailable; evaluation will use learning-to-rank scores.")
    print(type(exc).__name__, str(exc))

## 13. Safety filtering and MMR diversity selection

Before finalizing the list, the system removes candidates matching a configurable blocked-keyword list. It then uses Maximal Marginal Relevance (MMR) to choose up to 10 items, balancing relevance with semantic dissimilarity to items already selected. This keyword list is a basic guardrail, not a clinically validated safety system.

`mmr_select` greedily chooses the next item using both its relevance score and its semantic similarity to already selected items. `build_final_recommendations` blends cross-encoder and earlier scores, removes items flagged by the keyword guardrail, and uses MMR to select up to 10 final products.

In [ ]:
def mmr_select(items: Sequence[str], scores: Sequence[float], semantic: np.ndarray, ids,
               k: int, diversity: float = 0.20) -> List[str]:
    if not items:
        return []
    relevance = normalize_scores(scores)
    selected_indices: List[int] = []
    remaining = list(range(len(items)))
    while remaining and len(selected_indices) < k:
        best_index, best_value = None, -1e18
        for j in remaining:
            if not selected_indices:
                value = float(relevance[j])
            else:
                item_idx = ids.catalog2id[items[j]]
                selected_item_indices = [ids.catalog2id[items[s]] for s in selected_indices]
                similarities = [float(np.dot(semantic[item_idx], semantic[sidx])) for sidx in selected_item_indices]
                value = (1.0 - diversity) * float(relevance[j]) - diversity * max(similarities)
            if value > best_value:
                best_index, best_value = j, value
        selected_indices.append(best_index)
        remaining.remove(best_index)
    return [items[j] for j in selected_indices]

def build_final_recommendations(eval_df, ranked_items_by_user, ranked_scores_by_user,
                                histories, ids, semantic, metadata, cfg,
                                cross_scores_by_user=None):
    recs = {}
    rec_rows = []
    for row in eval_df.itertuples(index=False):
        items = ranked_items_by_user.get(row.user, [])
        original_scores = ranked_scores_by_user.get(row.user, [])
        base_lookup = {it: float(s) for it, s in zip(items, original_scores)}
        base_values = np.asarray([base_lookup.get(it, 0.0) for it in items], dtype=np.float32)
        base_normalized = normalize_scores(base_values)
        if cross_scores_by_user is not None:
            cross_lookup = {it: float(score) for it, score in cross_scores_by_user.get(row.user, [])}
            cross_values = np.asarray([cross_lookup.get(it, -1e9) for it in items], dtype=np.float32)
            cross_normalized = normalize_scores(cross_values)
            # Do not let a generic MS-MARCO cross-encoder erase all popularity,
            # semantic and collaborative signals learned earlier in the pipeline.
            blended = ((1.0 - cfg.cross_encoder_weight) * base_normalized
                       + cfg.cross_encoder_weight * cross_normalized)
            order = np.argsort(-blended)
            items = [items[i] for i in order]
            scores = blended[order].astype(float).tolist()
        else:
            order = np.argsort(-base_normalized)
            items = [items[i] for i in order]
            scores = base_normalized[order].astype(float).tolist()
        safe_items = [(it, score) for it, score in zip(items, scores) if is_safe(it, metadata, cfg)]
        safe_items = safe_items[:cfg.ranking_k]
        items = [x[0] for x in safe_items]
        scores = [x[1] for x in safe_items]
        final = mmr_select(items, scores, semantic, ids, cfg.final_k, diversity=cfg.mmr_diversity)
        final = [it for it in final if is_safe(it, metadata, cfg)][:cfg.final_k]
        recs[row.user] = final
        for rank, item in enumerate(final, start=1):
            rec_rows.append({
                "user": row.user,
                "rank": rank,
                "item": item,
                "title": metadata[item]["title"],
                "timestamp_target": int(row.timestamp),
            })
    return recs, pd.DataFrame(rec_rows)

## 14. Evaluation metrics and popularity baseline

We evaluate the single held-out target per user with Recall@10, MRR, and NDCG@10. We also report catalogue coverage, novelty, and intra-list diversity. The popularity baseline recommends high-scoring training items that the user has not already interacted with. These are offline metrics; this notebook does not run a live A/B experiment.

In [ ]:
def dcg(relevances: Sequence[float]) -> float:
    return float(sum(rel / math.log2(position + 2) for position, rel in enumerate(relevances)))

def recall_at_k(recommendations: Sequence[str], relevant: Sequence[str], k: int) -> float:
    return float(bool(set(recommendations[:k]) & set(relevant)))

def reciprocal_rank(recommendations: Sequence[str], relevant: Sequence[str]) -> float:
    relevant_set = set(relevant)
    for rank, item in enumerate(recommendations, start=1):
        if item in relevant_set:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(recommendations: Sequence[str], relevant: Sequence[str], k: int) -> float:
    rel_set = set(relevant)
    gains = [1.0 if item in rel_set else 0.0 for item in recommendations[:k]]
    actual = dcg(gains)
    ideal = dcg([1.0] * min(len(rel_set), k))
    return actual / ideal if ideal else 0.0

def calculate_metrics(eval_df, recs_by_user, train_df, ids, semantic, k):
    per_user = []
    for row in eval_df.itertuples(index=False):
        recs = recs_by_user.get(row.user, [])
        per_user.append({
            "user": row.user,
            "Recall@K": recall_at_k(recs, [row.item], k),
            "MRR": reciprocal_rank(recs, [row.item]),
            "NDCG@K": ndcg_at_k(recs, [row.item], k),
        })
    per_user_df = pd.DataFrame(per_user)
    flat = [item for recs in recs_by_user.values() for item in recs[:k]]
    catalog_coverage = len(set(flat)) / max(len(ids.id2catalog_item), 1)
    counts = train_df["item"].value_counts()
    total = max(int(counts.sum()), 1)
    n_items = max(len(counts), 1)
    novelty_values = []
    for item in flat:
        item_count = int(counts.get(item, 0))
        probability = (item_count + 1.0) / (total + n_items)
        novelty_values.append(-math.log2(probability))
    novelty_value = float(np.mean(novelty_values)) if novelty_values else 0.0

    diversity_values = []
    for recs in recs_by_user.values():
        item_indices = [ids.catalog2id[x] for x in recs[:k] if x in ids.catalog2id]
        if len(item_indices) < 2:
            continue
        vectors = semantic[item_indices]
        similarities = vectors @ vectors.T
        pair_similarities = similarities[np.triu_indices(len(item_indices), k=1)]
        diversity_values.append(float(np.mean(1.0 - pair_similarities)))
    ild = float(np.mean(diversity_values)) if diversity_values else 0.0

    metrics = {
        f"Recall@{k}": float(per_user_df["Recall@K"].mean()) if len(per_user_df) else 0.0,
        "MRR": float(per_user_df["MRR"].mean()) if len(per_user_df) else 0.0,
        f"NDCG@{k}": float(per_user_df["NDCG@K"].mean()) if len(per_user_df) else 0.0,
        f"CatalogCoverage@{k}": float(catalog_coverage),
        f"Novelty@{k}": float(novelty_value),
        f"IntraListDiversity@{k}": float(ild),
        "EvaluatedUsers": int(len(eval_df)),
    }
    return metrics, per_user_df

def popularity_recommendations(eval_df, histories, popularity, metadata, cfg):
    result = {}
    popularity_order = list(popularity.index)
    for row in eval_df.itertuples(index=False):
        seen = set(histories.get(row.user, []))
        recs = []
        for item in popularity_order:
            if item in seen or not is_safe(item, metadata, cfg):
                continue
            recs.append(item)
            if len(recs) >= cfg.final_k:
                break
        result[row.user] = recs
    return result

## 15. Run validation evaluation

This cell retrieves/ranks validation candidates, reports Candidate Recall@200 for the fused retrieval pool and each individual channel, optionally runs the cross-encoder, applies safety + MMR, and compares the hybrid list with the popularity baseline. Use validation metrics when choosing settings.

In [ ]:
metrics_out = {
    "project": "Health & Personal-Care Hybrid Recommendation System",
    "split_strategy": "per-user temporal: all but last two train, second-last validation, last test",
    "config": asdict(cfg),
    "dataset": {
        "cleaned_interactions_before_split": int(len(df)),
        "train_interactions": int(len(train_df)),
        "train_users": int(len(ids.id2user)),
        "train_catalogue_items": int(len(ids.id2item)),
        "semantic_candidate_catalogue_items": int(len(ids.id2catalog_item)),
        "validation_targets_retained": int(len(val_df)),
        "validation_targets_before_catalogue_filter": int(val_before),
        "test_targets_retained": int(len(test_df)),
        "test_targets_before_catalogue_filter": int(test_before),
    },
    "validation": {},
    "test": {},
}
per_user_frames = []

val_eval_df, val_ranked_items, val_profiles, val_ranked_scores = prepare_ranked_candidates(
    val_df, "validation", histories, ids, two_tower, semantic, index,
    semantic_index, cfg, popularity, metadata, feature_builder, ranker, scaler,
)
val_cross_scores = None
if cross_encoder is not None:
    try:
        val_cross_scores = cross_encoder_rerank_all(
            val_ranked_items, val_profiles, metadata, cross_encoder, batch_size=64
        )
    except Exception as exc:
        print("WARNING: Validation cross-encoder inference failed; using ranker/blended scores.")
        print(type(exc).__name__, str(exc))

val_recs, val_rec_frame = build_final_recommendations(
    val_eval_df, val_ranked_items, val_ranked_scores, histories, ids,
    semantic, metadata, cfg, cross_scores_by_user=val_cross_scores,
)
val_hybrid_metrics, val_hybrid_per_user = calculate_metrics(
    val_eval_df, val_recs, train_df, ids, semantic, cfg.final_k
)
val_hybrid_per_user["system"] = "hybrid"
val_hybrid_per_user["split"] = "validation"
val_baseline_recs = popularity_recommendations(val_eval_df, histories, popularity, metadata, cfg)
val_baseline_metrics, val_baseline_per_user = calculate_metrics(
    val_eval_df, val_baseline_recs, train_df, ids, semantic, cfg.final_k
)
val_baseline_per_user["system"] = "popularity_baseline"
val_baseline_per_user["split"] = "validation"
metrics_out["validation"] = {
    "hybrid": val_hybrid_metrics,
    "popularity_baseline": val_baseline_metrics,
}
per_user_frames.extend([val_hybrid_per_user, val_baseline_per_user])
val_rec_frame.to_csv(OUTPUT_DIR / "validation_recommendations.csv", index=False)
print("\nValidation hybrid metrics:")
print(json.dumps(val_hybrid_metrics, indent=2))
print("\nValidation popularity-baseline metrics:")
print(json.dumps(val_baseline_metrics, indent=2))

## 16. Run test evaluation

After the model and settings are fixed using training/validation data, evaluate the final pipeline on the held-out test targets. If you have already used the test results to tune retrieval weights, describe those results as exploratory rather than as an untouched final test.

In [ ]:
test_eval_df, test_ranked_items, test_profiles, test_ranked_scores = prepare_ranked_candidates(
    test_df, "test", histories, ids, two_tower, semantic, index,
    semantic_index, cfg, popularity, metadata, feature_builder, ranker, scaler,
)
test_cross_scores = None
if cross_encoder is not None:
    try:
        test_cross_scores = cross_encoder_rerank_all(
            test_ranked_items, test_profiles, metadata, cross_encoder, batch_size=64
        )
    except Exception as exc:
        print("WARNING: Test cross-encoder inference failed; using ranker/blended scores.")
        print(type(exc).__name__, str(exc))

test_recs, test_rec_frame = build_final_recommendations(
    test_eval_df, test_ranked_items, test_ranked_scores, histories, ids,
    semantic, metadata, cfg, cross_scores_by_user=test_cross_scores,
)
test_hybrid_metrics, test_hybrid_per_user = calculate_metrics(
    test_eval_df, test_recs, train_df, ids, semantic, cfg.final_k
)
test_hybrid_per_user["system"] = "hybrid"
test_hybrid_per_user["split"] = "test"
test_baseline_recs = popularity_recommendations(test_eval_df, histories, popularity, metadata, cfg)
test_baseline_metrics, test_baseline_per_user = calculate_metrics(
    test_eval_df, test_baseline_recs, train_df, ids, semantic, cfg.final_k
)
test_baseline_per_user["system"] = "popularity_baseline"
test_baseline_per_user["split"] = "test"
metrics_out["test"] = {
    "hybrid": test_hybrid_metrics,
    "popularity_baseline": test_baseline_metrics,
}
per_user_frames.extend([test_hybrid_per_user, test_baseline_per_user])
test_rec_frame.to_csv(OUTPUT_DIR / "test_recommendations.csv", index=False)
print("\nTest hybrid metrics:")
print(json.dumps(test_hybrid_metrics, indent=2))
print("\nTest popularity-baseline metrics:")
print(json.dumps(test_baseline_metrics, indent=2))

## 17. Save models, metrics, and recommendations

This cell writes the combined metrics, per-user metrics, and recommendations to `/content/artifacts`, then copies them to Google Drive so they persist after the Colab runtime resets.

In [ ]:
# Save combined metric outputs.
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metrics_out, indent=2), encoding="utf-8"
)
pd.concat([val_rec_frame.assign(split="validation"), test_rec_frame.assign(split="test")], ignore_index=True).to_csv(
    OUTPUT_DIR / "all_recommendations.csv", index=False
)
pd.concat(per_user_frames, ignore_index=True).to_csv(
    OUTPUT_DIR / "per_user_metrics.csv", index=False
)

model_metadata = {
    "architecture": "BPR MF + ID-based two-tower + Sentence Transformer semantic embeddings + hybrid HNSW/semantic exact FAISS/popularity retrieval with reciprocal rank fusion + pointwise HistGradientBoosting ranker + blended cross-encoder reranking + keyword safety guardrail + MMR",
    "embedding_model": cfg.embedding_model,
    "cross_encoder_model": cfg.cross_encoder_model if cross_encoder is not None else None,
    "cross_encoder_used": bool(cross_encoder is not None),
    "retrieval_k": cfg.retrieval_k,
    "ranking_k": cfg.ranking_k,
    "final_k": cfg.final_k,
    "important_limitations": [
        "Offline evaluation, not a live online A/B test.",
        "Keyword safety filtering is a basic guardrail and not medical validation.",
        "BPR treats every observed review as positive regardless of star rating.",
        "The pointwise ranker is not LambdaMART/listwise ranking.",
        "Cold-start and online production behavior require separate evaluation.",
    ],
}
(OUTPUT_DIR / "model_metadata.json").write_text(
    json.dumps(model_metadata, indent=2), encoding="utf-8"
)

DRIVE_OUTPUT = DRIVE_FOLDER / "final_artifacts"
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
print("Artifacts saved locally to:", OUTPUT_DIR)
print("Artifacts copied to Google Drive:", DRIVE_OUTPUT)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file()))

## 18. Display the final metric table and example recommendations

Use this cell after saving. It places validation and test results side by side for the hybrid system and popularity baseline, then displays the first 20 recommendation rows for inspection.

In [ ]:
rows = []
for split_name in ["validation", "test"]:
    for system_name, values in metrics_out[split_name].items():
        rows.append({"Split": split_name, "System": system_name, **values})
metrics_table = pd.DataFrame(rows)
display(metrics_table)
print("Dataset/split details:")
display(pd.DataFrame([metrics_out["dataset"]]))
print("Example test recommendations:")
display(test_rec_frame.head(20))

## 19. Download the artifacts ZIP

All final outputs are already copied to Google Drive. Optionally create a ZIP of the artifacts and download it to your computer for submission or backup.

In [ ]:
import shutil
from google.colab import files

zip_base = "/content/health_recommender_artifacts"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_DIR)
print("Created:", zip_path)
files.download(zip_path)